# InternVLA-N1 — official DualVLN agent

Hosted Gradio is down (401 / remote backend). This is InternNav `inference_only_demo.ipynb` plus a System-2 chat on **cuda:1**.
Kernel **internvla-n1**. No Habitat.

In [1]:
from pathlib import Path
import sys, glob
import numpy as np
from PIL import Image
from IPython.display import display, Markdown

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from patch_internnav import ensure_internnav
ensure_internnav()
from dualvln_play import DualVlnPlay
from system2_ask import System2Ask

candidates = [
    ROOT / "assets" / "realworld_sample_data1",
    ROOT / "vendor" / "InternNav" / "assets" / "realworld_sample_data1",
]
scene = next((p for p in candidates if p.exists()), None)
print("scene", scene)
if scene:
    instr_path = scene / "instruction.txt"
    instruction = instr_path.read_text().strip() if instr_path.exists() else "go to the kitchen"
    frames = sorted(glob.glob(str(scene / "debug_raw_*.jpg")))
else:
    instruction = "go to the kitchen"
    frames = []
print("instruction:", instruction)
print("n frames", len(frames))

patched /workspace/VLA_shopping/demo_sandboxes/internvla_n1/vendor/InternNav/setup.py python_requires -> >=3.8,<3.13
PROJECT_ROOT_PATH:/workspace/VLA_shopping/demo_sandboxes/internvla_n1/vendor/InternNav
internnav importable
scene /workspace/VLA_shopping/demo_sandboxes/internvla_n1/assets/realworld_sample_data1
instruction: Turn around and walk out of this office. Turn towards your slight right at the chair. Move forward to the walkway and go near the red bin. You can see an open door on your right side, go inside the open door. Stop at the computer monitor.
n frames 152


In [2]:
agent = DualVlnPlay()
print("DualVLN on cuda:0")

patched /workspace/VLA_shopping/demo_sandboxes/internvla_n1/vendor/InternNav/setup.py python_requires -> >=3.8,<3.13
internnav importable
args.model_path/workspace/VLA_shopping/demo_sandboxes/internvla_n1/weights/InternVLA-N1-DualVLN


xFormers not available
xFormers not available


flash_attention_2 failed, retrying with sdpa (official notebook allows this): [Errno 2] No such file or directory: 'checkpoints/depth_anything_v2_metric_hypersim_vits.pth'
args.model_path/workspace/VLA_shopping/demo_sandboxes/internvla_n1/weights/InternVLA-N1-DualVLN


FileNotFoundError: [Errno 2] No such file or directory: 'checkpoints/depth_anything_v2_metric_hypersim_vits.pth'

## Step the official agent

Reply = S2 `llm_output` (mid-level English you can read — not a conversation). Think log = pixel-goal + whether S2 ran this frame (`plan_step_gap`). The walk loop will not interview you. Free-form talk is the System-2 cell below.

In [ ]:
idx = 0
if frames:
    rgb = np.array(Image.open(frames[idx]).convert("RGB"))
else:
    rgb = np.zeros((480, 640, 3), dtype=np.uint8)
display(Image.fromarray(rgb))
out = agent.step(rgb, None, None, instruction)
display(Markdown("**Think**"))
print(out["think"])
display(Markdown("**Reply (S2)**"))
display(Markdown(out["reply"] or "_(empty this frame — S1 only)_"))
display(Image.fromarray(out["vis"]))

In [ ]:
# Advance a few official frames (S2 only every plan_step_gap).
if frames:
    for idx in range(1, min(5, len(frames))):
        rgb = np.array(Image.open(frames[idx]).convert("RGB"))
        out = agent.step(rgb, None, None, instruction)
        print(idx, "s2_ran", out["think"]["s2_ran"], "pixel", out["think"]["pixel_goal"])
        print("  S2:", (out["reply"] or "")[:160])
    display(Image.fromarray(out["vis"]))

## Ask System 2 (Qwen2.5-VL card on cuda:1)

In [ ]:
s2 = System2Ask()
ask = s2.ask(Image.fromarray(rgb), "What are you looking at, and what should you do next to follow the instruction?")
display(Markdown("**Reply**"))
display(Markdown(ask["reply"]))